# Install Required Libraries

In [1]:
!pip install yt-dlp
!pip install openai-whisper
!pip install pydub
!pip install youtube-search-python
!apt-get install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.1/172.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 17.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 65.4 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20240930-py3-none-any.whl size=803321 sha256=e33f622a89ce3e13c188283431ccb3687073797f76920adc884432ce4e439e68
  Stored in directory: /root/.cache/pip/wheels/dd/4a/1f/d1c4bf3b9133c8168fe617ed979cab7b14fe381d059ffb9d83
Successfully built openai-whisper
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.5/99.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 6.5 MB/s eta 0:00:

# Import Libraries

In [2]:
import yt_dlp
import whisper
import os
import json
import pandas as pd
from pydub import AudioSegment
from youtubesearchpython import VideosSearch
from tqdm.notebook import tqdm
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# Create Directories for Storage

In [3]:
os.makedirs("audio_files", exist_ok=True)
os.makedirs("transcriptions", exist_ok=True)
os.makedirs("processed_data", exist_ok=True)

# Gather 500 Arabic YouTube Video URLs

In [4]:
search_keywords = [
    "الاقتصاد العالمي", "القضايا الاجتماعية", "الأحداث الجارية"
    "المؤتمرات العالمية", "الدراسات الدولية", "السياحة العالمية",
    "السفر حول العالم", "الدورات المجانية", "منح دراسية", "المنح التعليمية",
    "الدورات الاحترافية", "برمجة الألعاب", "الواقع الافتراضي",
    "الواقع المعزز", "إنترنت الأشياء", "البلوكتشين", "التعدين",
    "العملات الرقمية", "الشبكات العصبية", "التعلم الآلي", "الأدوات البرمجية",
    "الهندسة الكهربائية", "الهندسة الميكانيكية", "الطب العام",
    "الطاقة الشمسية", "الطاقة المتجددة", "الأدوات الطبية", "التكنولوجيا الحديثة"
]


target_video_count = 200
collected_video_urls = []

min_duration = 5 * 60
max_duration = 10 * 60

while len(collected_video_urls) < target_video_count:
    keyword = random.choice(search_keywords)

    try:
        videos_search = VideosSearch(keyword, limit=20)
        results = videos_search.result()

        if results and 'result' in results:
            for video in results['result']:
                if 'id' in video and 'duration' in video and video['duration']:
                    try:
                        duration_parts = list(map(int, video['duration'].split(':')))
                        duration_seconds = sum(x * 60 ** i for i, x in enumerate(reversed(duration_parts)))

                        if min_duration <= duration_seconds <= max_duration:
                            video_url = f"https://www.youtube.com/watch?v={video['id']}"

                            if video_url not in collected_video_urls:
                                collected_video_urls.append(video_url)
                                if len(collected_video_urls) >= target_video_count:
                                    break
                    except ValueError:
                        print(f"Skipping video with unexpected duration format: {video['duration']}")
                else:
                    print(f"Skipping video missing 'id' or 'duration': {video}")
        else:
            print(f"No valid results returned for keyword '{keyword}'. Trying a different keyword.")

    except TypeError as e:
        print(f"TypeError encountered: {e}. Retrying with a different keyword.")
    except Exception as e:
        print(f"Unexpected error with keyword '{keyword}': {e}. Retrying.")

    time.sleep(2)

print(f"Collected {len(collected_video_urls)} video URLs with duration between 5 and 10 minutes.")

TypeError encountered: 'NoneType' object is not iterable. Retrying with a different keyword.
TypeError encountered: 'NoneType' object is not iterable. Retrying with a different keyword.
TypeError encountered: 'NoneType' object is not iterable. Retrying with a different keyword.
TypeError encountered: 'NoneType' object is not iterable. Retrying with a different keyword.
TypeError encountered: 'NoneType' object is not iterable. Retrying with a different keyword.
TypeError encountered: 'NoneType' object is not iterable. Retrying with a different keyword.
TypeError encountered: 'NoneType' object is not iterable. Retrying with a different keyword.
TypeError encountered: 'NoneType' object is not iterable. Retrying with a different keyword.
TypeError encountered: 'NoneType' object is not iterable. Retrying with a different keyword.
TypeError encountered: 'NoneType' object is not iterable. Retrying with a different keyword.
TypeError encountered: 'NoneType' object is not iterable. Retrying wit

#  Download YouTube Audio and Extract Metadata

In [5]:
def download_youtube_audio(video_url, output_dir="audio_files"):
    try:
        ydl_opts = {
            'format': 'bestaudio/best',
            'outtmpl': f'{output_dir}/%(id)s.%(ext)s',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'wav',
                'preferredquality': '192',
            }],
            'quiet': True,
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(video_url, download=True)
            audio_path = os.path.join(output_dir, f"{info['id']}.wav")
            metadata = {
                'id': info['id'],
                'title': info.get('title', ''),
                'duration': info.get('duration', 0),
                'url': video_url
            }
            return audio_path, metadata
    except Exception as e:
        print(f"Error downloading {video_url}: {e}")
        return None, None


# Transcribe Audio Using Whisper

In [6]:
def transcribe_audio_whisper(audio_path, model="base"):
    try:
        whisper_model = whisper.load_model(model)
        result = whisper_model.transcribe(audio_path)
        transcription = result['text']
        return transcription
    except Exception as e:
        print(f"Error transcribing {audio_path}: {e}")
        return None


# Process Each Video

In [7]:
def process_video_data(video_url, model="base"):
    audio_path, metadata = download_youtube_audio(video_url)
    if audio_path and metadata:
        transcription = transcribe_audio_whisper(audio_path, model)
        if transcription:
            # Save transcription as JSON
            transcription_path = f"transcriptions/{metadata['id']}.json"
            with open(transcription_path, 'w') as f:
                json.dump({
                    "id": metadata['id'],
                    "title": metadata['title'],
                    "duration": metadata['duration'],
                    "url": metadata['url'],
                    "transcription": transcription
                }, f, ensure_ascii=False)
            return {
                "id": metadata['id'],
                "title": metadata['title'],
                "duration": metadata['duration'],
                "url": metadata['url'],
                "transcription": transcription,
                "audio_path": audio_path
            }
    return None

# Run the Pipeline to Process the Videos

In [8]:
dataset = []

for url in tqdm(collected_video_urls):
    data = process_video_data(url, model="base")
    if data:
        dataset.append(data)

  0%|          | 0/200 [00:00<?, ?it/s]


  0%|                                               | 0.00/139M [00:00<?, ?iB/s]
  4%|█▌                                    | 5.75M/139M [00:00<00:02, 60.3MiB/s]
  8%|███▏                                  | 11.5M/139M [00:00<00:02, 54.5MiB/s]
 12%|████▌                                 | 16.7M/139M [00:00<00:02, 44.0MiB/s]
 15%|█████▊                                | 21.1M/139M [00:00<00:02, 42.9MiB/s]
 18%|██████▉                               | 25.3M/139M [00:00<00:03, 30.9MiB/s]
 21%|███████▊                              | 28.6M/139M [00:00<00:04, 24.7MiB/s]
 23%|████████▌                             | 31.4M/139M [00:01<00:04, 23.9MiB/s]
 26%|█████████▋                            | 35.5M/139M [00:01<00:03, 28.2MiB/s]
 30%|███████████▎                          | 41.3M/139M [00:01<00:02, 36.0MiB/s]
 34%|████████████▉                         | 47.0M/139M [00:01<00:02, 41.9MiB/s]
 39%|██████████████▊                       | 54.0M/139M [00:01<00:01, 49.7MiB/s]
 43%|████████████████▏     

/usr/local/lib/python3.10/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)


/usr/local/lib/python3.10/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)


/usr/local/lib/python3.10/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)


...

/usr/local/lib/python3.10/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)


/usr/local/lib/python3.10/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)


/usr/local/lib/python3.10/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)


/usr/local/lib/python3.10/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)


/usr/local/lib/python3.10/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)


# Save the Dataset as a CSV File

In [9]:
df = pd.DataFrame(dataset)
df.to_csv("processed_data/arabic_tts_dataset.csv", index=False)

print("Data collection and transcription process is complete.")

Data collection and transcription process is complete.
